# VoteAnomaly · exploración de la participación

Objetivo: describir la participación y comprobar el detector estadístico en una muestra
de 12 mesas simuladas de `tests/fixtures/sample_election.csv`. La muestra sirve para
verificar el método. El conjunto del proyecto se genera por separado.

Instala las dependencias de `requirements.txt` y abre este notebook con el kernel
Python 3 de ese entorno. Ejecuta todas las celdas en orden.

## Preparar la muestra

La muestra original tenía dos candidatos. Se trasladaron sus identificadores de mesa
a `record_id` y `station_id`, se agregó un cero para el candidato C y se usó
`null_votes` para los votos nulos. Se conservan las cifras originales.

Usamos el preprocesamiento del equipo. La participación es el cociente entre los
votos reportados y el censo cuando el censo es positivo. Las tasas se expresan como
decimales y se conservan los valores finitos fuera de [0, 1].

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd

fixture = Path("tests/fixtures/sample_election.csv")
cwd = Path.cwd()
root = next((parent for parent in (cwd, *cwd.parents) if (parent / fixture).is_file()), None)
if root is None:
    raise FileNotFoundError(f"Abre el notebook desde el repositorio que contiene {fixture}.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.analysis import summarize_data
from src.detection.statistical import detect_statistical_anomalies
from src.preprocessing import BALLOT_COLUMNS, preprocess_dataset

prepared = preprocess_dataset(pd.read_csv(root / fixture))
registered = prepared["registered_voters"]
prepared["participation_rate"] = prepared["total_votes"] / registered.where(registered.gt(0))
prepared[["record_id", "registered_voters", "total_votes", "participation_rate"]]

,record_id,registered_voters,total_votes,participation_rate
0,M001,200,142,0.710000
1,M002,180,114,0.633333
2,M003,220,149,0.677273
3,M004,250,174,0.696000
4,M005,160,104,0.650000
5,M006,240,156,0.650000
6,M007,200,230,1.150000
7,M008,210,154,0.733333
8,M009,190,123,0.647368
9,M010,260,174,0.669231


## Estadísticas de participación

Los conteos distinguen tasas finitas, faltantes, infinitas e inválidas. Las estadísticas
usan solo tasas finitas. Los cuartiles se interpolan linealmente y la desviación estándar
es muestral. Los valores indefinidos aparecen como `null` en el JSON.

In [2]:
summary = summarize_data(prepared)
print(json.dumps(summary, indent=2, allow_nan=False))

{
  "record_count": 12,
  "finite_participation_count": 12,
  "missing_participation_count": 0,
  "nonfinite_participation_count": 0,
  "invalid_participation_count": 0,
  "over_100_percent_count": 1,
  "participation_rate": {
    "mean": 0.715103705547978,
    "median": 0.6732517482517482,
    "min": 0.6333333333333333,
    "max": 1.15,
    "q1": 0.65,
    "q3": 0.7024999999999999,
    "std": 0.14010162702024645
  }
}


## Alertas IQR

El rango intercuartílico es IQR = Q3 − Q1. Con al menos cuatro tasas finitas, el detector
alerta por debajo de Q1 − 1.5 × IQR o por encima de Q3 + 1.5 × IQR. Una tasa igual a un
límite queda dentro. Si IQR es cero, los límites coinciden y toda tasa finita distinta
del valor común de los cuartiles queda fuera. La detección usa únicamente la tasa.

In [3]:
alerts = detect_statistical_anomalies(prepared)
print(alerts.to_string(index=False))

record_id      method                                                               reason
     M007 statistical participation_rate 1.15 is above IQR upper bound 0.7812499999999998.


## Interpretación y límites

M007 permite revisar una participación superior al censo. M008 permite comprobar si
una discordancia entre el total reportado y la suma de las papeletas llama la atención
del IQR, que observa la participación pero no inspecciona la suma.

In [4]:
for record_id in ("M007", "M008"):
    row = prepared.loc[prepared["record_id"].eq(record_id)].iloc[0]
    ballot_sum = row[list(BALLOT_COLUMNS)].sum()
    flagged = alerts["record_id"].eq(record_id).any()
    print(
        f"{record_id} | participación {row['participation_rate']:.2%}"
        f" | total reportado {row['total_votes']} | suma de papeletas {ballot_sum}"
        f" | alerta IQR: {'sí' if flagged else 'no'}"
    )

M007 | participación 115.00% | total reportado 230 | suma de papeletas 230 | alerta IQR: sí
M008 | participación 73.33% | total reportado 154 | suma de papeletas 124 | alerta IQR: no


Los límites dependen de la distribución de la muestra. Con participación legítimamente
distinta entre grupos, una mesa válida puede quedar fuera de los límites. Con anomalías
frecuentes, los cuartiles pueden desplazarse. Conviene contrastar las alertas con reglas
de consistencia y evaluar su cobertura por separado. Una alerta señala un valor inusual
dentro de estos datos simulados y por sí sola no permite concluir que hubo fraude.